# Prévision des sinistres mensuels en assurance auto avec PROC FORECAST

## Résumé exécutif

Une équipe actuarielle de provisionnement a besoin d'une visibilité à 12 mois sur le nombre mensuel de sinistres auto, montrant à la fois une croissance régulière du portefeuille et une nette hausse hivernale. Ce notebook génère cinq années de nombres de sinistres mensuels synthétiques (60 mois, janvier 2020 - décembre 2024, allant d'environ 1 460 à 2 780 sinistres), puis utilise **PROC FORECAST** pour ajuster une base autorégressive pas à pas et un modèle saisonnier de Holt-Winters multiplicatif. Chaque modèle produit 12 prévisions ponctuelles mensuelles avec des limites de confiance à 95 % pour la planification de la capacité et des réserves. Le modèle saisonnier de Holt-Winters projette la demande la plus forte un à deux mois en avance (environ 2 780-2 900 sinistres), diminuant vers un creux autour de l'étape neuf (environ 2 200), tandis que la base autorégressive projette une décroissance plus lisse ; les deux bandes de confiance s'élargissent avec l'horizon.

## Sources de données

| Ensemble de données | Lignes | Granularité | Variables clés | Description |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Une ligne par mois calendaire (janvier 2020 - décembre 2024) | `date` (date SAS, `MONYY7.`), `claim_count` (numérique) | Nombres de sinistres auto mensuels synthétiques construits à partir d'une tendance de croissance linéaire (~9 sinistres/mois), d'un cycle annuel sinusoïdal, d'une hausse hivernale additive en déc./janv./févr., et d'un bruit gaussien (`rand('normal')`). La graine `20240531` la rend reproductible. |

# Prévision des sinistres mensuels en assurance auto

Les équipes de provisionnement et de planification de capacité d'un assureur particuliers ont besoin d'une vision prospective du nombre de **sinistres auto** qui seront déclarés chaque mois. Le volume de sinistres de ce portefeuille croît régulièrement à mesure que le portefeuille s'étend, et il augmente fortement chaque hiver lorsque le verglas, la neige et la réduction de la luminosité font grimper les sinistres de collision et de bris de glace.

Ce notebook présente un flux de travail complet avec `PROC FORECAST` :

1. Générer une série de sinistres mensuels synthétique réaliste.
2. Ajuster une base **autorégressive pas à pas (STEPAR)** qui capture la tendance et l'autocorrélation.
3. Ajuster un modèle **de Holt-Winters multiplicatif (WINTERS)** qui modélise explicitement le cycle saisonnier de 12 mois.
4. Comparer les deux modèles et interpréter la prévision prospective et la bande de confiance.

Tout s'exécute sur des données synthétiques intégrées — aucun fichier externe ni accès réseau.

## Étape 1 — Générer la série synthétique de sinistres

L'étape DATA ci-dessous construit **60 observations mensuelles** (janvier 2020 à décembre 2024). Pour chaque mois, nous combinons quatre composantes qui reflètent un portefeuille auto réel :

- **Tendance** — une base d'environ 1 800 sinistres augmentant d'environ 9 par mois à mesure que l'exposition croît.
- **Cycle annuel** — un terme sinus/cosinus donnant une onde saisonnière lisse.
- **Hausse hivernale** — un supplément additif en décembre, janvier et février.
- **Bruit** — `rand('normal', 0, 90)` pour une variabilité mensuelle réaliste.

`call streaminit()` fixe le flux aléatoire afin que le notebook soit reproductible. La variable `date` est une véritable date SAS construite avec `INTNX` et formatée `MONYY7.`, et l'instruction `ID date INTERVAL=MONTH` la désigne comme identifiant temporel. Les 14 premières lignes affichées ci-dessous se situent entre environ 1 460 et 2 450 sinistres.

In [1]:
DONNÉES claims;
    APPELER streaminit(20240531);
    FAIRE i = 0 JUSQU_À 59;
        /* Construire une vraie date SAS mensuelle pour que INTERVAL=MONTH s'aligne */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = janv ... 12 = déc */

        trend  = 1800 + 9 * i;               /* base d'exposition croissante */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx DANS (12, 1, 2));   /* hausse verglas/neige */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        SI claim_count < 0 ALORS claim_count = 0;
        SORTIE;
    FIN;
    GARDER date claim_count;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=claims(obs=14) ÉTIQUETTE;
    ÉTIQUETTE date = 'Mois' claim_count = 'Sinistres déclarés';
    TITRE 'Premiers 14 mois de sinistres auto synthétiques';
EXÉCUTER;

                                    Premiers 14 mois de sinistres auto synthétiques                                     

  Obs   Mois    Sinistres déclarés
    1  21915                  2178
    2  21946                  2281
    3  21975                  2252
    4  22006                  1974
    5  22036                  2067
    6  22067                  1805
    7  22097                  1697
    8  22128                  1619
    9  22159                  1462
   10  22189                  1688
   11  22220                  1713
   12  22250                  2008
   13  22281                  2116
   14  22312                  2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Étape 2 — Base autorégressive pas à pas (METHOD=STEPAR)

`METHOD=STEPAR` est la méthode par défaut. Elle ajuste d'abord un modèle de tendance temporelle — ici `TREND=2` pour une tendance linéaire — puis applique une **autorégression pas à pas** sur les résidus, en intégrant et en conservant les décalages AR selon leur significativité. `AR=4` plafonne l'ordre autorégressif candidat à quatre décalages, ce qui est largement suffisant pour une série mensuelle à momentum de court terme.

Options clés utilisées ici :

- `LEAD=12` — prévoir 12 mois au-delà des données.
- `ALPHA=0.05` — limites de confiance à 95 %.
- `OUTFULL` — empiler les lignes historiques `ACTUAL` avec les lignes de l'horizon de prévision dans l'ensemble `OUT=` (plutôt que les lignes de prévision seules).
- `OUTLIMIT` — ajouter les **colonnes** de limites de confiance inférieure/supérieure (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — nomme l'identifiant temporel et déclare que la série est mensuelle.

In [2]:
PROCÉDURE forecast DONNÉES=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=fc_stepar(obs=10) ÉTIQUETTE;
    ÉTIQUETTE date = 'Mois' claim_count = 'Sinistres' _type_ = 'Type'
          l95_claim_count = 'Limite inf. 95%' u95_claim_count = 'Limite sup. 95%';
    TITRE 'Résultats STEPAR : lignes réelles, ajustées et prévisionnelles';
EXÉCUTER;

                                    Premiers 14 mois de sinistres auto synthétiques                                     

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                             Résultats STEPAR : lignes réelles, ajustées et prévisionnelles                             

  Obs   Mois    Type    Sinistres  Limite inf. 95%  Limite sup. 95%
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Lecture de l'ensemble `OUT=`

L'ensemble `OUT=` empile les lignes selon une colonne `_TYPE_` et porte les limites de confiance sous forme de **colonnes** annexes :

| Élément | Nature | Signification |
|---------|------|-------------|
| `_TYPE_ = 'ACTUAL'` | ligne | La valeur observée de `claim_count` pour chacun des 60 mois historiques. |
| `_TYPE_ = 'FORECAST'` | ligne | Les 12 prévisions ponctuelles pour l'horizon de prévision. |
| `L95_claim_count` / `U95_claim_count` | colonne | Limites de confiance inférieure / supérieure à 95 %, renseignées sur les lignes `FORECAST` (manquantes sur les lignes `ACTUAL`). Le niveau numérique reflète `ALPHA=`. |

L'ensemble `OUT=` imprimé ici contient donc 72 lignes : 60 lignes historiques `ACTUAL` suivies de 12 lignes d'horizon `FORECAST`. Les dix premières lignes montrées ci-dessus sont toutes `ACTUAL`, les colonnes de limites de confiance étant manquantes puisque les limites ne s'appliquent qu'aux lignes de prévision.

> **Remarque moteur.** Le `OUTFULL` de SAS écrit également une ligne `FORECAST` intra-échantillon à un pas en avance pour chaque mois historique, et `OUTRESID` ajoute des lignes `_TYPE_='RESIDUAL'`. Le PROC FORECAST de Jenner n'émet pas encore ces lignes ajustées/résiduelles intra-échantillon (suivi par le test de lacune `400979`), donc ce notebook lit uniquement l'historique `ACTUAL` et l'horizon prospectif `FORECAST`.

## Étape 3 — Modèle saisonnier de Holt-Winters (METHOD=WINTERS)

Le modèle STEPAR capture la tendance et l'autocorrélation de court terme, mais ne modélise pas explicitement la hausse récurrente de décembre à février. Pour une série au rythme annuel marqué, le **Holt-Winters multiplicatif** est l'outil le plus adapté.

`METHOD=WINTERS` décompose la série en niveau, tendance linéaire (`TREND=2`), et un **facteur saisonnier multiplicatif**. `SEASONS=12` déclare un cycle saisonnier de 12 périodes (mensuel). Nous redemandons un `LEAD` de 12 mois avec des limites à 95 % et `OUTFULL OUTLIMIT` afin que la sortie s'aligne avec l'exécution STEPAR.

Comme le moteur avance l'`ID` de prévision d'une unité par étape (plutôt que de respecter `INTERVAL=MONTH` pour les dates de l'horizon — test de lacune `400979`), la cellule ci-dessous examine la prévision par **mois à venir (étape 1-12)** plutôt que de se fier aux dates calendaires affichées.

In [3]:
PROCÉDURE forecast DONNÉES=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
EXÉCUTER;

/* Conserver l'horizon prospectif de 12 mois et l'indexer par étape (1..12). */
DONNÉES horizon;
    DÉFINIR fc_winters;
    RETENIR months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    GARDER months_ahead claim_count l95_claim_count u95_claim_count;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=horizon ÉTIQUETTE;
    ÉTIQUETTE months_ahead     = 'Mois à venir'
          claim_count       = 'Sinistres prévus'
          l95_claim_count   = 'Limite inf. 95%'
          u95_claim_count   = 'Limite sup. 95%';
    TITRE 'Prévision prospective Holt-Winters à 12 mois (par étape)';
EXÉCUTER;

                             Résultats STEPAR : lignes réelles, ajustées et prévisionnelles                             

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                Prévision prospective Holt-Winters à 12 mois (par étape)                                

  Obs   Mois à venir   Sinistres prévus  Limite inf. 95%  Limite sup. 95%
    1              1         2783.07951      2577.844742      2988.314278
    2              2        2897.355557      2607.109764      3187.601349
    3              3        2805.272075      2449.795029       3160.74912
    4              4        2664.498049      2254.028514      3074.967585
    5              5        2628.810136      2169.891244      3087.729029
    6              6         2548.39319      2045.672732      3051.113649
    7              7        2391.649756        1848.6496      2934.649912
    8              8        2239.948352      1659.45676


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Étape 4 — Comparer les deux modèles face à face

La façon la plus claire de juger si le modèle saisonnier apporte une réelle valeur ajoutée est de placer sa prévision à 12 mois à côté de la base STEPAR, étape par étape. Nous extrayons les lignes `FORECAST` des deux ensembles `OUT=`, indexons chacune par mois à venir, et les fusionnons afin que la divergence soit visible d'un coup d'œil.

Les deux méthodes s'accordent sur le niveau global mais divergent sur la **forme** : Holt-Winters reporte le rythme annuel dans l'horizon (un niveau plus élevé en début d'horizon qui diminue vers un creux à mi-horizon puis remonte), tandis que STEPAR — qui ne modélise la saisonnalité que de façon indirecte via les décalages AR — décroît plus régulièrement vers la moyenne de la série. L'écart entre les deux à chaque étape est le signal saisonnier que STEPAR laisse de côté.

> SAS piloterait normalement ce contrôle d'adéquation avec les résidus à un pas en avance `OUTRESID` (`_TYPE_='RESIDUAL'`). Jenner ne renseigne pas encore ces lignes (test de lacune `400979`), nous comparons donc directement les deux prévisions prospectives — un diagnostic qui n'utilise que la sortie réellement produite par le moteur.

In [4]:
/* Horizon STEPAR, indexé par mois à venir. */
DONNÉES stepar_h;
    DÉFINIR fc_stepar;
    RETENIR months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    GARDER months_ahead stepar;
EXÉCUTER;

/* Horizon WINTERS, indexé par mois à venir. */
DONNÉES winters_h;
    DÉFINIR fc_winters;
    RETENIR months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    GARDER months_ahead winters;
EXÉCUTER;

/* Fusionner les deux et montrer l'écart saisonnier que STEPAR ne capture pas. */
DONNÉES compare;
    FUSIONNER stepar_h winters_h;
    PAR months_ahead;
    seasonal_gap = winters - stepar;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=compare ÉTIQUETTE;
    ÉTIQUETTE months_ahead = 'Mois à venir'
          stepar        = 'Prévision STEPAR'
          winters       = 'Prévision Winters'
          seasonal_gap  = 'Winters - STEPAR';
    format stepar winters seasonal_gap 8.0;
    TITRE 'STEPAR contre Holt-Winters : comparaison des prévisions à 12 mois';
EXÉCUTER;

                           STEPAR contre Holt-Winters : comparaison des prévisions à 12 mois                            

  Obs   Mois à venir   Prévision STEPAR   Prévision Winters  Winters - STEPAR
    1              1               2619                2783               164
    2              2               2537                2897               361
    3              3               2384                2805               421
    4              4               2214                2664               450
    5              5               2089                2629               540
    6              6               2010                2548               539
    7              7               1982                2392               410
    8              8               1995                2240               245
    9              9               2031                2197               166
   10             10               2075                2354               280
   11             11


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Étape 5 — Prévoir toutes les lignes d'activité en une fois (traitement BY)

Les cycles de provisionnement réels couvrent plusieurs produits. Avec les données triées par `line_of_business`, une instruction `BY` permet à `PROC FORECAST` d'ajuster un **modèle saisonnier indépendant pour chaque groupe** en un seul appel. Ici, nous synthétisons un portefeuille Auto (volume de base plus élevé) et un portefeuille Habitation (volume de base plus faible) et prévoyons les deux à six mois. `OUTALL` écrit l'ensemble complet des lignes — l'historique `ACTUAL` et l'horizon `FORECAST` — avec les colonnes de limites pour chaque groupe, et nous ne conservons que les lignes `FORECAST` pour le tableau ci-dessous.

In [5]:
DONNÉES multi_book;
    APPELER streaminit(20240531);
    LONGUEUR line_of_business $10;
    FAIRE lob = 1 JUSQU_À 2;
        SI lob = 1 ALORS line_of_business = 'Auto';
        SINON            line_of_business = 'Habitation';
        FAIRE i = 0 JUSQU_À 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi DANS (12, 1, 2))
                + rand('normal', 0, 70));
            SORTIE;
        FIN;
    FIN;
    GARDER line_of_business date claim_count;
EXÉCUTER;

PROCÉDURE TRIER DONNÉES=multi_book;
    PAR line_of_business date;
EXÉCUTER;

PROCÉDURE forecast DONNÉES=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    PAR line_of_business;
    id date interval=month;
    VAR claim_count;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=fc_by(obs=12) ÉTIQUETTE;
    ÉTIQUETTE line_of_business = "Ligne d'activité"
          date             = 'Mois'
          claim_count      = 'Sinistres'
          _type_           = 'Type'
          l95_claim_count  = 'Limite inf. 95%'
          u95_claim_count  = 'Limite sup. 95%';
    OÙ _type_ = 'FORECAST';
    TITRE "Prévisions à 6 mois par ligne d'activité";
EXÉCUTER;

                           STEPAR contre Holt-Winters : comparaison des prévisions à 12 mois                            


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Habitation

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                        Prévisions à 6 mois par ligne d'activité                                        

  Obs   Ligne d'activité   Mois      Type    Sinistres  Limite inf. 95%  Limite sup. 95%  RESIDUAL_CLAIM_COUNT
    1  Auto               23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Auto               23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Auto               23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Auto             


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Habitation
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interprétation des résultats

**Ce que les modèles indiquent à l'équipe de provisionnement :**

- **STEPAR** reproduit la dérive haussière et le momentum de court terme, mais sa prévision à 12 mois décroît régulièrement d'environ 2 620 (étape 1) vers environ 1 980 à mi-horizon avant de remonter vers environ 2 140 — elle ne reproduit pas de pic hivernal marqué, car elle ne traite la saisonnalité que via les décalages autorégressifs. C'est une base rapide raisonnable.
- **WINTERS** avec `SEASONS=12` reporte directement le rythme annuel via ses facteurs saisonniers multiplicatifs : sa prévision est la plus élevée en début d'horizon (environ 2 780 à l'étape 1, environ 2 900 à l'étape 2), diminue vers un creux proche de l'étape 9 (environ 2 200), puis remonte à l'étape 12 (environ 2 800). Par rapport à la base STEPAR, elle se situe **150 à 660 sinistres plus haut** sur la majeure partie de l'horizon (voir la colonne `Winters - STEPAR` de l'étape 4) — cet écart est le signal saisonnier que le modèle autorégressif laisse de côté.
- La **bande de confiance à 95 %** (colonnes `L95_*`/`U95_*`, contrôlées par `ALPHA=`) s'élargit à mesure que l'horizon s'étend — pour le modèle WINTERS, d'une largeur d'environ 410 sinistres à l'étape 1 à environ 1 420 à l'étape 12 — signal honnête que les estimations de fin d'horizon comportent davantage d'incertitude que celles de court terme. Les provisionneurs devraient constituer leur capital par rapport à la limite supérieure, pas seulement à la prévision ponctuelle.
- Le **traitement BY** produit les prévisions Auto et Habitation en un seul passage, chacune avec son propre ajustement saisonnier. Le portefeuille Auto prévoit dans une fourchette d'environ 2 510 à 2 600, tandis que le portefeuille Habitation se situe près de 1 870 à 2 030 — chaque ligne conserve ainsi son propre niveau et sa propre forme saisonnière, un schéma que l'équipe pourrait étendre à chaque produit du portefeuille.

**En résumé :** pour une série de nombres de sinistres au cycle annuel marqué, `METHOD=WINTERS SEASONS=12` est le choix le plus naturel ; la base STEPAR est un contrôle de cohérence utile, et `OUTFULL`/`OUTLIMIT` combinés à une comparaison de modèles étape par étape permettent à l'actuaire de dimensionner la réserve prospective avec sa bande d'incertitude.

> **Remarque sur la fidélité du moteur.** Ce notebook documente deux limitations actuelles du PROC FORECAST de Jenner (test de lacune `400979`) : l'`ID` de l'horizon de prévision est avancé d'une unité par étape plutôt que par `INTERVAL=MONTH`, si bien que les dates de prévision affichées ne correspondent pas aux mois calendaires 2025 prévus (nous examinons donc l'horizon par étape) ; et `OUTRESID`/`OUTALL` ne renseignent pas encore les lignes `_TYPE_='RESIDUAL'` à un pas en avance, si bien que les diagnostics résiduels sont remplacés par une comparaison directe des deux modèles. Les prévisions ponctuelles et les limites de confiance elles-mêmes sont bien produites et sont ce sur quoi s'appuie le récit ci-dessus.